# Historical capstone notebook
This notebook comes from the 2025 research workspace. Outputs, execution counts, and cell metadata have been removed for public distribution. It requires authorized access to the original PostgreSQL database. See `../docs/reproduction.md` for dependencies and limitations. The public offline demo is `../scripts/run_demo.py`.


In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import gc
from datetime import datetime

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
import matplotlib.pyplot as plt

import psycopg2
import yaml
from sqlalchemy import create_engine

In [ ]:
import os, yaml
from sqlalchemy import create_engine

# Try env var first (no file with secrets needed)
db_url = os.getenv("DATABASE_URL")

if not db_url:
    # Fall back to a local file: database.yaml (preferred) or database.local.yaml
    for candidate in ("database.yaml", "database.local.yaml"):
        if os.path.exists(candidate):
            with open(candidate, "r") as f:
                cfg = yaml.safe_load(f)
            db_url = f"postgresql://{cfg['user']}:{cfg['pass']}@{cfg['host']}:{cfg['port']}/{cfg['db']}"
            break

if not db_url:
    raise FileNotFoundError(
        "No DATABASE_URL env var and no database.yaml/database.local.yaml found."
    )

engine = create_engine(db_url)

# Example SQL query to fetch rescues data from PostgreSQL
base_rescues = pd.read_sql("SELECT * FROM rescues;", engine)


Features we need:
- When is the rescue published
- Weather on that day
- Date in a week, hour in a day
- Distance from user to rescue, from user to donor
- user/rescue/donor experience 
    - count of past rescues
    - registeration time
    - past (user, rescue), (user, donor), (rescue, donor) pairs
- Recurrence
- Average volunteer rating at recipient/donor
- This volunteer's past average rating
- item weight
- food type
- vehicle type
- Longitude/Latitude/Elevation
- County Type (zone)
- recipient location population type
- is contact available or not
- is location photo available or not <- not available in photos.csv, actually can match with contact info

In [ ]:
query = """
SELECT * 
FROM public.rescues
WHERE published_at <= '2022-12-27'
"""
base_rescues = pd.read_sql(query, engine)

In [ ]:
base_rescues.shape

In [ ]:
def gen_base():
    # Load base data from PostgreSQL instead of CSV
    query_rescues = """
    SELECT id AS rescue_id, donation_id, recipient_location_id, published_at, completed_at, user_id, volunteer_rating, volunteer_comment
    FROM rescues
    WHERE published_at <= '2022-12-27' AND completed_at <= '2022-12-27';
    """
    base_rescues = pd.read_sql(query_rescues, engine)
    base_rescues[['published_at', 'completed_at']] = base_rescues[['published_at', 'completed_at']].apply(pd.to_datetime)
    
    query_routes = """
        SELECT id AS routes_id, published_at, user_id, completed_at, donation_id, organization_id, volunteer_rating, volunteer_comment
        FROM routes
        WHERE published_at <= '2022-12-27' AND completed_at <= '2022-12-27';
    """
    base_routes = pd.read_sql(query_routes, engine)
    base_routes[['published_at', 'completed_at']] = base_routes[['published_at', 'completed_at']].apply(pd.to_datetime)


    # This code is essentially stacking the two tables on top of each other
    # Concatenate the two dataframes
    base = pd.concat([base_rescues, base_routes], axis=0)

    # Sort by 'published_at' and reset index
    base = base.sort_values(by='published_at', ascending=True).reset_index(drop=True)

    # Cleanup
    del base_rescues
    del base_routes
    gc.collect()

    return base

# Call the function to generate the base dataset
base = gen_base()
#base = base[(base['published_at'] >= '2022-01-01') & (base['published_at'] < '2022-06-01')]
print(base.shape)

print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())


In [ ]:
def gen_publish_date(base):
    # Filter for only the first month of January
    # Add discrete, one-hot-encoded features for the date and time
    base['published_Y'] = base['published_at'].dt.year
    base['published_M'] = base['published_at'].dt.month
    base['published_D'] = base['published_at'].dt.day
    base['published_H'] = base['published_at'].dt.hour
    return base

# Call the function and print the shape to confirm filtering
base = gen_publish_date(base)
print(base.shape)

print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

#There is the same amount of Nan values in published_at as the published_X 
#columns because the published_at columns have Nans from the routes table


In [ ]:
def gen_weather(base):
    # Fetch weather data from PostgreSQL instead of a CSV file
    query_weather = """
    SELECT *
    FROM derivative.weather
    WHERE "STATION" = 72520094823
    AND ("REPORT_TYPE" = 'FM-12' OR "REPORT_TYPE" = 'FM-15' OR "REPORT_TYPE" = 'FM-16')
    AND "DATE" <= '2022-12-27';
    """
    weather = pd.read_sql(query_weather, engine)


    weather_features = ['HourlyDewPointTemperature',
                        'HourlyDryBulbTemperature',
                        'HourlyPrecipitation', # fill na with 0
                        'HourlyRelativeHumidity',
                        'HourlyStationPressure',
                        'HourlyVisibility',
                        'HourlyWetBulbTemperature',
                        'HourlyWindSpeed',]

    # Ensure the selected columns are numeric and handle missing data
    for wf in weather_features:
        weather[wf] = pd.to_numeric(weather[wf], errors='coerce')
    
    weather['DATE'] = pd.to_datetime(weather['DATE'])
    weather['HourlyPrecipitation'] = weather['HourlyPrecipitation'].fillna(0.0)
    weather.reset_index(inplace=True)

    # 1-NN impute NA values by date
    for wf in weather_features:
        na_ind = np.argwhere(weather[wf].isna().values == True).flatten()
        for i in na_ind:
            dateNA = weather.loc[i, 'DATE']
            time_delta = abs((weather['DATE'] - dateNA).values)
            candidates = np.argsort(time_delta)[1:]
            assert(len(candidates) > 0)
            not_found = True
            j = 0
            while not_found:
                nearest_idx = candidates[j]
                nearest_val = weather.loc[nearest_idx, wf]
                if not np.isnan(nearest_val):
                    weather.loc[i, wf] = nearest_val
                    not_found = False
                else:
                    j += 1
    
    # Associate weather data with the rescue data by the nearest date
    for wf in weather_features:
        splits = 101
        CHUNK_SIZE = base.shape[0] // (splits - 1)
        wf_all_nearest = np.zeros((base.shape[0],))
        rescue_dates = np.array(base['published_at'], dtype=np.datetime64)
        all_weather_dates = np.array(weather['DATE'], dtype=np.datetime64).reshape(-1, 1)
        for i in range(splits):
            print(wf, i)
            rescue_date = rescue_dates[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE]
            time_delta = abs((all_weather_dates - rescue_date))
            nearest_idx = np.argmin(time_delta, axis=0)
            nearest_val = weather.loc[nearest_idx, wf]
            wf_all_nearest[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE] = nearest_val
            gc.collect()
        base[wf] = wf_all_nearest
        gc.collect()
    
    return base

base = gen_weather(base)
base.to_csv('./all_rescues_info.csv')

print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

In [ ]:
base = pd.read_csv('./all_rescues_info.csv')
print(base.shape)

In [ ]:
# recipient info

def onehot(df, in_feature_name, id_name):
    feat_values = np.unique(df[in_feature_name]) # one recipient_location_id <-> multiple new feature
    collect_res = {id_name : np.unique(df[id_name])}
    for x in feat_values:
        collect_res[f'{in_feature_name}_{x}'] = np.zeros((len(np.unique(df[id_name])),))
    # "N-hot"
    for i, rec in enumerate(tqdm(collect_res[id_name])):
        all_feats = list(df[df[id_name] == rec][in_feature_name])
        val_feat, count_feat = np.unique(all_feats, return_counts=True)
        for j, x in enumerate(val_feat):
            collect_res[f'{in_feature_name}_{x}'][i] += count_feat[j]
    collect_res = pd.DataFrame(collect_res)   
    return collect_res

def process_recipient():
    # Fetching from PostgreSQL instead of CSV
    recipient_locations = pd.read_sql("""
        SELECT id AS recipient_location_id, recipient_id, address_id, phone AS recipient_phone, recurrence_id 
        FROM recipient_locations
    """, engine)

    recipients = pd.read_sql("""
        SELECT id AS recipient_location_id, created_at AS recipient_created_at 
        FROM recipients
    """, engine)

    addresses = pd.read_sql("""
        SELECT id AS address_id, longitude AS recipient_longitude, latitude AS recipient_latitude, 
               neighborhood AS recipient_neighborhood, county AS recipient_county
        FROM addresses
    """, engine)

    recipient_location_households = pd.read_sql("""
        SELECT household_id, recipient_location_id 
        FROM recipient_location_households
    """, engine)

    households = pd.read_sql("""
        SELECT id AS household_id, size AS recipient_household_size 
        FROM households
    """, engine)

    # Replacing pd.read_csv with SQL queries for the one-hot encoded tables
    recipient_locations_food_restrictions = pd.read_sql("""
        SELECT recipient_location_id, food_restriction_id 
        FROM recipient_locations_food_restrictions
    """, engine)

    recipient_locations_food_types = pd.read_sql("""
        SELECT recipient_location_id, food_type_id 
        FROM recipient_locations_food_types
    """, engine)

    recipient_locations_nonprofit = pd.read_sql("""
        SELECT recipient_location_id, nonprofit_category_id 
        FROM recipient_locations_nonprofit_categories
    """, engine)

    recipient_locations_population = pd.read_sql("""
        SELECT recipient_location_id, population_type_id 
        FROM recipient_locations_population_types
    """, engine)

    # Perform the merges and one-hot encoding
    merged_households = pd.merge(households, recipient_location_households, on=['household_id'], how='left')
    collect_recipient_households = onehot(merged_households,'recipient_household_size', 'recipient_location_id')

    collect_recipient_food_restrictions = onehot(recipient_locations_food_restrictions, 'food_restriction_id','recipient_location_id')
    collect_recipient_food_types = onehot(recipient_locations_food_types, 'food_type_id','recipient_location_id')
    collect_recipient_nonprofit = onehot(recipient_locations_nonprofit, 'nonprofit_category_id','recipient_location_id')
    collect_recipient_population = onehot(recipient_locations_population, 'population_type_id','recipient_location_id')

    # Merging everything together
    merged_recipient = pd.merge(recipient_locations, recipients, on=['recipient_location_id'], how='left')
    merged_recipient = pd.merge(merged_recipient, addresses, on=['address_id'], how='left')
    merged_recipient = pd.merge(merged_recipient, collect_recipient_households, on=['recipient_location_id'], how='left')
    merged_recipient = pd.merge(merged_recipient, collect_recipient_food_restrictions, on=['recipient_location_id'], how='left')
    merged_recipient = pd.merge(merged_recipient, collect_recipient_food_types, on=['recipient_location_id'], how='left')
    merged_recipient = pd.merge(merged_recipient, collect_recipient_nonprofit, on=['recipient_location_id'], how='left')
    merged_recipient = pd.merge(merged_recipient, collect_recipient_population, on=['recipient_location_id'], how='left')
    merged_recipient = merged_recipient.drop(columns=['recipient_location_id','address_id'])

    assert(merged_recipient.shape[0] == recipient_locations.shape[0])

    return merged_recipient

merged_recipient = process_recipient()
print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

In [ ]:
def process_donors():
    # Fetch data from PostgreSQL using SQL queries
    donations = pd.read_sql("""
        SELECT id AS donation_id, donor_location_id 
        FROM donations
    """, engine)

    donor_locations = pd.read_sql("""
        SELECT id AS donor_location_id, donor_id, address_id, phone AS donor_phone 
        FROM donor_locations
    """, engine)

    addresses = pd.read_sql("""
        SELECT id AS address_id, longitude AS donor_longitude, latitude AS donor_latitude, 
               neighborhood AS donor_neighborhood, county AS donor_county 
        FROM addresses
    """, engine)

    donors = pd.read_sql("""
        SELECT id AS donor_id, created_at AS donor_created_at 
        FROM donors
    """, engine)

    donation_items = pd.read_sql("""
        SELECT donation_id, quantity, food_id 
        FROM donation_items
    """, engine)

    # Now replace the one-hot encoding logic with the donation items loaded from the DB
    id_name = 'donation_id'
    in_feature_name = 'food_id'
    feat_values = np.unique(donation_items[in_feature_name]) 
    collect_res = {id_name: np.unique(donation_items[id_name])}
    
    for x in feat_values:
        collect_res[f'{in_feature_name}_{x}'] = np.zeros((len(np.unique(donation_items[id_name])),))
    
    collect_res['total_quantity'] = np.zeros((len(np.unique(donation_items[id_name])),))
    
    # "one-hot"
    for i, rec in enumerate(tqdm(collect_res[id_name])):
        all_feats = list(donation_items[donation_items[id_name] == rec][in_feature_name])
        # Sum all quantities
        agg_quantities = sum(list(donation_items[donation_items[id_name] == rec]['quantity']))
        collect_res['total_quantity'][i] = agg_quantities
        val_feat, count_feat = np.unique(all_feats, return_counts=True)
        for j, x in enumerate(val_feat):
            collect_res[f'{in_feature_name}_{x}'][i] += count_feat[j]
    
    collect_res = pd.DataFrame(collect_res)
    collect_donation_items = collect_res

    # Perform the merges
    merged_donor = pd.merge(donations, donor_locations, on=['donor_location_id'], how='left')
    merged_donor = pd.merge(merged_donor, addresses, on=['address_id'], how='left')
    merged_donor = pd.merge(merged_donor, donors, on=['donor_id'], how='left')
    merged_donor = pd.merge(merged_donor, collect_donation_items, on=['donation_id'], how='left')

    # Drop unnecessary columns
    merged_donor = merged_donor.drop(columns=['donor_location_id', 'donor_id', 'address_id'])

    assert(merged_donor.shape[0] == donations.shape[0])

    return merged_donor

# Now call the function to process donors
merged_donor = process_donors()
print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

In [ ]:
def process_users():
    # Fetch data from PostgreSQL using SQL queries
    users = pd.read_sql("""
        SELECT id AS user_id, address_id, created_at AS user_created_at, phone AS user_phone 
        FROM users
    """, engine)

    addresses = pd.read_sql("""
        SELECT id AS address_id, longitude AS user_longitude, latitude AS user_latitude, 
               neighborhood AS user_neighborhood, county AS user_county 
        FROM addresses
    """, engine)

    user_vehicles = pd.read_sql("""
        SELECT user_id, vehicle_type 
        FROM user_vehicles
    """, engine)

    # One-hot encoding for user vehicles
    collect_user_vehicles = onehot(user_vehicles, 'vehicle_type', 'user_id')

    # Perform the merges
    merged_users = pd.merge(users, addresses, on=['address_id'], how='left')
    merged_users = pd.merge(merged_users, collect_user_vehicles, on=['user_id'], how='left')

    # Drop unnecessary columns
    merged_users = merged_users.drop(columns=['address_id'])

    assert(merged_users.shape[0] == users.shape[0])
    
    
    return merged_users

# Now call the function to process users
merged_users = process_users()
print(merged_users.columns)
print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

In [ ]:
merged_recipient = merged_recipient.rename(columns={'recipient_id':'recipient_location_id'})
print(merged_recipient.columns)

In [ ]:
base = pd.merge(base, merged_recipient, on=['recipient_location_id'], how='left')
print("After merging with merged_recipient:", base.shape)
base = pd.merge(base, merged_donor, on=['donation_id'], how='left')
print("After merging with merged_donor:", base.shape)
base = pd.merge(base, merged_users, on=['user_id'], how='left')
print("After merging with merged_users:", base.shape)


print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

#There are all the Nans for vehicle type because of the fact that when you merge left with base, there are some ids
#that aren't in the user_vehicles table. This means that for some ids we can't assign a vehicle

In [ ]:
# interactions
def L2(x1, y1, x2, y2):
    return ((x1-x2)**2 + (y1-y2)**2)**0.5

import math

def haversine_distance(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a)) 
    r = 6371 # Radius of earth
    return c * r

base['user2donor'] = L2(base['user_longitude'], base['user_latitude'],
                                base['donor_longitude'], base['donor_latitude'])
base['user2recipient'] = L2(base['user_longitude'], base['user_latitude'],
                            base['recipient_longitude'], base['recipient_latitude'])

print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())
print(base['published_at'])


In [ ]:

# --- 1) Diagnose the columns that have NaNs ---
print(base[['user_id','donation_id','recipient_location_id','rescue_id']].isna().sum())

# --- 2) Fill missing values with a placeholder ---
base['user_id'] = base['user_id'].fillna(-1)
base['donation_id'] = base['donation_id'].fillna(-1)
base['recipient_location_id'] = base['recipient_location_id'].fillna(-1)
base['rescue_id'] = base['rescue_id'].fillna(-1)

# --- 3) Sort by published_at ---
base = base.sort_values('published_at')

# If any old columns exist, drop them
cols_to_drop = [
    'donor_past_counts', 'user_past_counts', 'recipient_past_counts',
    'user_recipient_past_counts', 'user_donor_past_counts',
    'donor_recipient_past_counts', 'user_donor_recipient_past_counts'
]
for c in cols_to_drop:
    if c in base.columns:
        base.drop(columns=c, inplace=True)

# --- 4) Donor Past Counts ---
base["_donor_first_seen"] = ~base.groupby("donation_id")["rescue_id"] \
                                 .transform(lambda x: x.duplicated())
base["donor_past_counts"] = (
    base.groupby("donation_id")["_donor_first_seen"]
    .cumsum()
    .shift(fill_value=0)
)

# --- 5) User Past Counts ---
base["_user_first_seen"] = ~base.groupby("user_id")["rescue_id"] \
                                .transform(lambda x: x.duplicated())
base["user_past_counts"] = (
    base.groupby("user_id")["_user_first_seen"]
    .cumsum()
    .shift(fill_value=0)
)

# --- 6) Recipient Past Counts ---
base["_recipient_first_seen"] = ~base.groupby("recipient_location_id")["rescue_id"] \
                                     .transform(lambda x: x.duplicated())
base["recipient_past_counts"] = (
    base.groupby("recipient_location_id")["_recipient_first_seen"]
    .cumsum()
    .shift(fill_value=0)
)

# --- 7) User–Recipient Past Counts ---
base["_user_recipient_first_seen"] = ~base.groupby(["user_id","recipient_location_id"])["rescue_id"] \
                                          .transform(lambda x: x.duplicated())
base["user_recipient_past_counts"] = (
    base.groupby(["user_id","recipient_location_id"])["_user_recipient_first_seen"]
    .cumsum()
    .shift(fill_value=0)
)

# --- 8) User–Donor Past Counts ---
base["_user_donor_first_seen"] = ~base.groupby(["user_id","donation_id"])["rescue_id"] \
                                      .transform(lambda x: x.duplicated())
base["user_donor_past_counts"] = (
    base.groupby(["user_id","donation_id"])["_user_donor_first_seen"]
    .cumsum()
    .shift(fill_value=0)
)

# --- 9) Donor–Recipient Past Counts ---
base["_donor_recipient_first_seen"] = ~base.groupby(["donation_id","recipient_location_id"])["rescue_id"] \
                                          .transform(lambda x: x.duplicated())
base["donor_recipient_past_counts"] = (
    base.groupby(["donation_id","recipient_location_id"])["_donor_recipient_first_seen"]
    .cumsum()
    .shift(fill_value=0)
)

# --- 10) User–Donor–Recipient Past Counts ---
base["_user_donor_recipient_first_seen"] = ~base.groupby(["user_id","donation_id","recipient_location_id"])["rescue_id"] \
                                                .transform(lambda x: x.duplicated())
base["user_donor_recipient_past_counts"] = (
    base.groupby(["user_id","donation_id","recipient_location_id"])["_user_donor_recipient_first_seen"]
    .cumsum()
    .shift(fill_value=0)
)

# Clean up helper columns
base.drop(columns=[
    "_donor_first_seen", "_user_first_seen", "_recipient_first_seen",
    "_user_recipient_first_seen", "_user_donor_first_seen", 
    "_donor_recipient_first_seen", "_user_donor_recipient_first_seen"
], inplace=True)

print(base[[
    'donor_past_counts', 'user_past_counts', 'recipient_past_counts',
    'user_recipient_past_counts', 'user_donor_past_counts',
    'donor_recipient_past_counts', 'user_donor_recipient_past_counts'
]].head(10))





In [ ]:
print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

# Keep your original column creations and checks:
rows = base.shape[0]
base['avg_past_user_rating'] = [0]*rows
base['avg_past_recipient_rating'] = [0]*rows
base['avg_past_donor_rating'] = [0]*rows

# Check if base DataFrame has rows
if rows > 0:
    print(f"Number of rows in base: {rows}")
else:
    print("Warning: The base DataFrame is empty!")

##############################################################################
# 1) Sort by 'published_at' so "previous" rows are actually above the current row
##############################################################################
base = base.sort_values('published_at')

##############################################################################
# 2) For each of user_id / donation_id / recipient_location_id, compute:
#    - The cumulative sum of volunteer_rating so far (excluding the current row)
#    - The cumulative count of non-NaN ratings so far (excluding the current row)
#    Then average = (cum_sum / cum_count), fill missing with 3.
#
#    Why does this replicate "base.loc[r, 'published_at'] < curr_time"?
#    Because after sorting by 'published_at', every row above is guaranteed
#    to have an earlier (or equal) timestamp. We then SHIFT by 1 position
#    so we exclude the current row's rating from its own average.
##############################################################################

#---------- Helper for each grouping column ----------
def set_past_average(base, groupcol, outcol):
    """
    For each group in `groupcol`, compute the average volunteer_rating
    of all strictly earlier rows, then store in `outcol`.
    Defaults to 3 if no prior rating exists.
    """
    # Count how many non-NaN ratings have appeared so far in this group
    base[f"__tmp_count_{outcol}"] = (
        base['volunteer_rating']
        .notna()
        .groupby(base[groupcol])
        .cumsum()
    )
    # Sum of ratings (treat NaN as 0 so it doesn't poison the sum)
    base[f"__tmp_sum_{outcol}"] = (
        base['volunteer_rating']
        .fillna(0)
        .groupby(base[groupcol])
        .cumsum()
    )

    # SHIFT by 1 row within each group so the current row
    # does not include its own rating:
    base[f"__tmp_count_{outcol}"] = base.groupby(groupcol)[f"__tmp_count_{outcol}"].shift(fill_value=0)
    base[f"__tmp_sum_{outcol}"]   = base.groupby(groupcol)[f"__tmp_sum_{outcol}"].shift(fill_value=0)

    # Average = (cumulative sum of prior ratings) / (# of prior non-NaN ratings)
    base[outcol] = base[f"__tmp_sum_{outcol}"] / base[f"__tmp_count_{outcol}"]
    # If the count was zero, we get NaN → fill with 3
    base[outcol].fillna(3, inplace=True)


#---------- 2a) avg_past_user_rating ----------
set_past_average(base, groupcol='user_id', outcol='avg_past_user_rating')

#---------- 2b) avg_past_donor_rating ----------
set_past_average(base, groupcol='donation_id', outcol='avg_past_donor_rating')

#---------- 2c) avg_past_recipient_rating ----------
set_past_average(base, groupcol='recipient_location_id', outcol='avg_past_recipient_rating')


##############################################################################
# 3) (Optional) Drop the temporary columns used for cumsum / counts
#    so your final DataFrame has only the original + the three new columns.
##############################################################################
temp_cols = [c for c in base.columns if c.startswith('__tmp_')]
base.drop(columns=temp_cols, inplace=True)

##############################################################################
# 4) Print the column of interest if it exists
##############################################################################
if 'avg_past_recipient_rating' in base.columns:
    print(base['avg_past_recipient_rating'])
else:
    print("Column 'avg_past_recipient_rating' does not exist in base.")

print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())


In [ ]:
# time until Jun 1, 2023
end_date = datetime(2023, 6, 1, 0, 0, 0)

#end_date = datetime(2022, 5, 31, 23, 59, 59)

base['donor_exp'] = [0] * rows
base['recipient_exp'] = [0] * rows
base['user_exp'] = [0] * rows
 
for r in tqdm(range(rows)):
    try:
        donor_date_str = base.loc[r, 'donor_created_at'].split('.')[0]
        donor_intvl = end_date - datetime.fromisoformat(donor_date_str)
        base.loc[r, 'donor_exp'] = int(donor_intvl.total_seconds())
    except:
        base.loc[r, 'donor_exp'] = np.nan

    try:
        recip_date_str = base.loc[r, 'recipient_created_at'].split('.')[0]
        recip_intvl = end_date - datetime.fromisoformat(recip_date_str)
        base.loc[r, 'recipient_exp'] = int(recip_intvl.total_seconds())
    except:
        base.loc[r, 'recipient_exp'] = np.nan

    try:
        user_date_str = base.loc[r, 'user_created_at'].split('.')[0]
        user_intvl = end_date - datetime.fromisoformat(user_date_str)
        base.loc[r, 'user_exp'] = int(user_intvl.total_seconds())
    except:
        base.loc[r, 'user_exp'] = np.nan


print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

if 'avg_past_recipient_rating' in base.columns:
    print(base['avg_past_recipient_rating'])
else:
    print("Column 'avg_past_recipient_rating' does not exist in base.")

In [ ]:
print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())

# Keep your original column creations and checks:
rows = base.shape[0]
base['avg_past_user_rating'] = [0]*rows
base['avg_past_recipient_rating'] = [0]*rows
base['avg_past_donor_rating'] = [0]*rows

# Check if base DataFrame has rows
if rows > 0:
    print(f"Number of rows in base: {rows}")
else:
    print("Warning: The base DataFrame is empty!")

##############################################################################
# 1) Sort by 'published_at' so "previous" rows are actually above the current row
##############################################################################
base = base.sort_values('published_at')

##############################################################################
# 2) For each of user_id / donation_id / recipient_location_id, compute:
#    - The cumulative sum of volunteer_rating so far (excluding the current row)
#    - The cumulative count of non-NaN ratings so far (excluding the current row)
#    Then average = (cum_sum / cum_count), fill missing with 3.
#
#    Why does this replicate "base.loc[r, 'published_at'] < curr_time"?
#    Because after sorting by 'published_at', every row above is guaranteed
#    to have an earlier (or equal) timestamp. We then SHIFT by 1 position
#    so we exclude the current row's rating from its own average.
##############################################################################

#---------- Helper for each grouping column ----------
def set_past_average(base, groupcol, outcol):
    """
    For each group in `groupcol`, compute the average volunteer_rating
    of all strictly earlier rows, then store in `outcol`.
    Defaults to 3 if no prior rating exists.
    """
    # Count how many non-NaN ratings have appeared so far in this group
    base[f"__tmp_count_{outcol}"] = (
        base['volunteer_rating']
        .notna()
        .groupby(base[groupcol])
        .cumsum()
    )
    # Sum of ratings (treat NaN as 0 so it doesn't poison the sum)
    base[f"__tmp_sum_{outcol}"] = (
        base['volunteer_rating']
        .fillna(0)
        .groupby(base[groupcol])
        .cumsum()
    )

    # SHIFT by 1 row within each group so the current row
    # does not include its own rating:
    base[f"__tmp_count_{outcol}"] = base.groupby(groupcol)[f"__tmp_count_{outcol}"].shift(fill_value=0)
    base[f"__tmp_sum_{outcol}"]   = base.groupby(groupcol)[f"__tmp_sum_{outcol}"].shift(fill_value=0)

    # Average = (cumulative sum of prior ratings) / (# of prior non-NaN ratings)
    base[outcol] = base[f"__tmp_sum_{outcol}"] / base[f"__tmp_count_{outcol}"]
    # If the count was zero, we get NaN → fill with 3
    base[outcol].fillna(3, inplace=True)


#---------- 2a) avg_past_user_rating ----------
set_past_average(base, groupcol='user_id', outcol='avg_past_user_rating')

#---------- 2b) avg_past_donor_rating ----------
set_past_average(base, groupcol='donation_id', outcol='avg_past_donor_rating')

#---------- 2c) avg_past_recipient_rating ----------
set_past_average(base, groupcol='recipient_location_id', outcol='avg_past_recipient_rating')


##############################################################################
# 3) (Optional) Drop the temporary columns used for cumsum / counts
#    so your final DataFrame has only the original + the three new columns.
##############################################################################
temp_cols = [c for c in base.columns if c.startswith('__tmp_')]
base.drop(columns=temp_cols, inplace=True)

##############################################################################
# 4) Print the column of interest if it exists
##############################################################################
if 'avg_past_recipient_rating' in base.columns:
    print(base['avg_past_recipient_rating'])
else:
    print("Column 'avg_past_recipient_rating' does not exist in base.")

print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())



In [ ]:
base.to_csv('all_rescues_info-noorg.csv',index=False)

In [ ]:
# base = pd.read_csv('all_rescues_info-noorg.csv')

# if 'avg_past_recipient_rating' in base.columns:
#     print(base['avg_past_recipient_rating'])
# else:
#     print("Column 'avg_past_recipient_rating' does not exist in base.")

In [ ]:
base['user_hasphone'] = base['user_phone'].isnull()
base['donor_hasphone'] = base['donor_phone'].isnull()
base['recipient_hasphone'] = base['recipient_phone'].isnull()
base['is_recurrence'] = base['recurrence_id'].isnull()

print("Shape of base after filtering:", base.shape)
print("Number of NaNs in each column after filtering:")
print(base.isna().sum())



In [ ]:
continuous_feat = ['HourlyDewPointTemperature','HourlyDryBulbTemperature','HourlyPrecipitation',
                   'HourlyRelativeHumidity','HourlyStationPressure','HourlyVisibility','HourlyWetBulbTemperature',
                    'HourlyWindSpeed',
                    'recipient_longitude','recipient_latitude',
                    'donor_longitude','donor_latitude','total_quantity',
                    'user_longitude','user_latitude','user2donor','user2recipient',
                    'donor_exp', 'recipient_exp','user_exp',
                    'avg_past_user_rating','avg_past_recipient_rating','avg_past_donor_rating',
                    'published_Y','published_M','published_D','published_H']
discrete_feat = []

onehot_prefix = [# 'donor_neighborhood','donor_county','user_neighborhood','user_county',
                # 'recipient_neighborhood','recipient_county',
                'recipient_household_size',
                'food_restriction_id','food_type_id','nonprofit_category_id',
                'population_type_id','food_id','vehicle_type']
for f in list(base.columns):
    f_remove_suffix = "_".join(f.split("_")[:-1])
    if f_remove_suffix in onehot_prefix:
        discrete_feat.append(f)

discrete_feat.extend(['user_hasphone','donor_hasphone','recipient_hasphone','is_recurrence'])
features = continuous_feat + discrete_feat

Use volunteer rating as proxy

In [ ]:
# Print columns to see types and identify numeric columns if needed
print(base.columns.tolist())

# Select the relevant columns including 'published_at' and 'volunteer_rating'
data = base[features + ['published_at', 'volunteer_rating']]

# Drop rows where 'volunteer_rating' is NaN
data = data.dropna(subset=['volunteer_rating'])

# Convert 'published_at' to datetime to ensure compatibility with date operations
data['published_at'] = pd.to_datetime(data['published_at'])

# Fill NaNs only in numeric columns
numeric_data = data.select_dtypes(include=[np.number])  # Select only numeric columns
data[numeric_data.columns] = numeric_data.fillna(numeric_data.median())  # Fill NaNs in numeric columns only

# Now `data` should have NaNs filled only in numeric columns
print(data.head())

In [ ]:
data.shape

In [ ]:
base['volunteer_rating'].isnull().sum()

In [ ]:
from sklearn import preprocessing
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

In [ ]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42, sampling_strategy='auto')

In [ ]:
# build model!
train_val_split = datetime(2020, 1, 1, 0, 0, 0)
# data['published_at'] = pd.to_datetime(data['published_at'])
# train_dataset = data[data['published_at'] < train_val_split]
# val_dataset = data[data['published_at'] >= train_val_split]
# train_dataset = train_dataset[features + ['published_at','volunteer_rating']]
# val_dataset = val_dataset[features + ['published_at','volunteer_rating']]

#train_val_split = datetime(2022, 3, 1, 0, 0, 0)
print("Date range:", data['published_at'].min(), "to", data['published_at'].max())
data['published_at'] = pd.to_datetime(data['published_at'])

# Split data based on the new date
train_dataset = data[data['published_at'] < train_val_split]
val_dataset = data[data['published_at'] >= train_val_split]

# Select only necessary features and target for train and validation datasets
train_dataset = train_dataset[features + ['published_at', 'volunteer_rating']]
val_dataset = val_dataset[features + ['published_at', 'volunteer_rating']]

print("Train Dataset:\n", train_dataset)
print("Validation Dataset:\n", val_dataset)

In [ ]:
# train, test set
X_train = np.array(train_dataset[features])
y_train = np.array(train_dataset['volunteer_rating'],dtype=np.uint8)
print("Train Label distribution:",np.unique(y_train, return_counts=True))
X_test = np.array(val_dataset[features])
y_test = np.array(val_dataset['volunteer_rating'],dtype=np.uint8)
print("Val Label distribution:",np.unique(y_test, return_counts=True))

print("X_test:\n", X_test)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
 
  
# creating the dataset
data = {'1':2604, '2':1560, '3':4143,'4':57039}
courses = list(data.keys())
values = list(data.values())
  
fig = plt.figure(figsize = (5, 5))
 
# creating the bar plot
plt.bar(courses, values, color ='navy',
        width = 0.4)
 
plt.xlabel("Ratings")
plt.ylabel("Count")
plt.title("Train istribution of volunteer ratings")
plt.show()

In [ ]:
print("X_test:\n", X_test)

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#changed
# Step 1: Ensure X_train and X_test are numeric arrays
X_train = np.array(X_train, dtype=np.float64)
X_test = np.array(X_test, dtype=np.float64)

# Step 2: Handle NaNs: Impute and keep consistent columns in both X_train and X_test
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Step 3: Normalize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train and X_test after imputation and scaling:")
print("X_train:\n", X_train)
print("X_test:\n", X_test)

# Step 4: Model training
models = {
    'lgb': lgb.LGBMClassifier(random_state=42),
    'lr': LogisticRegression(random_state=42, max_iter=500)  # Increased max_iter
}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predict_labels = model.predict(X_test)
    predict_proba = model.predict_proba(X_test)
    acc = accuracy_score(y_test, predict_labels)
    f1 = f1_score(y_test, predict_labels, average="macro")
    roc = roc_auc_score(y_test, predict_proba, multi_class="ovr", average="macro")
    print(f"{model_name} - acc: {acc:.3f}, f1: {f1:.3f}, roc: {roc:.3f}")

# Optional: Use Lasso (L1) for feature selection in Logistic Regression
reduced_model = LogisticRegression(random_state=42, penalty='l1', solver='saga', max_iter=5000)
reduced_model.fit(X_train, y_train)

# Extract non-zero coefficients to select important features
non_zero_mask = reduced_model.coef_[0] != 0

valid_columns_mask = ~np.isnan(X_train).all(axis=0)
adjusted_features = np.array(features)
if len(adjusted_features) > X_train.shape[1]:
    # Truncate `features` to match the actual number of columns in X_train
    adjusted_features = adjusted_features[:X_train.shape[1]]
# Use non_zero_mask on adjusted_features
preserved_features = adjusted_features[non_zero_mask]

print('Reduced model features:', preserved_features)

# Evaluate reduced model
reduced_predict_labels = reduced_model.predict(X_test)
reduced_predict_proba = reduced_model.predict_proba(X_test)
reduced_acc = accuracy_score(y_test, reduced_predict_labels)
reduced_f1 = f1_score(y_test, reduced_predict_labels, average="macro")
reduced_roc = roc_auc_score(y_test, reduced_predict_proba, multi_class="ovr", average="macro")
print(f'Reduced Model - acc: {reduced_acc:.3f}, f1: {reduced_f1:.3f}, roc: {reduced_roc:.3f}')

In [ ]:
X_res, y_res = ros.fit_resample(X_train, y_train)
np.unique(y_res, return_counts=True)

clf = lgb.LGBMClassifier(random_state=42)
clf = clf.fit(X_res, y_res)
predict_labels = clf.predict(X_test)
predict_proba = clf.predict_proba(X_test)
acc = accuracy_score(y_test, predict_labels)
f1 = f1_score(y_test, predict_labels, average="macro")
roc = roc_auc_score(y_test, predict_proba, multi_class="ovr", average="macro")
print(f'acc: {acc:.3f}, f1: {f1:.3f}, roc: {roc:.3f}')

cm = confusion_matrix(y_test, predict_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=clf.classes_)
disp.plot()
plt.show()

#importance_map = pd.DataFrame({'Feature' : features, 'Value' : clf.feature_importances_})
#changed
valid_columns_mask = ~np.isnan(X_train).all(axis=0)

# Make sure we only apply the mask up to the length of 'features'
min_dim = min(len(features), len(valid_columns_mask))

# Slice both lists/masks to the same length
features = features[:min_dim]
valid_columns_mask = valid_columns_mask[:min_dim]

adjusted_features = np.array(features)[valid_columns_mask]

# Ensure adjusted_features has the same length as clf.feature_importances_
if len(adjusted_features) == len(clf.feature_importances_):
    importance_map = pd.DataFrame({'Feature': adjusted_features, 'Value': clf.feature_importances_})
else:
    print("Mismatch in length between features and feature importances.")
    # Optionally: Adjust the code to handle this scenario gracefully
    importance_map = pd.DataFrame({'Feature': adjusted_features[:len(clf.feature_importances_)], 'Value': clf.feature_importances_})

plt.figure(figsize=(20, 10))
sns.barplot(x="Value", y="Feature", data=importance_map.sort_values(by="Value", ascending=False)[:30])
plt.title('LightGBM Features')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# train, test set
X_train_123 = np.array(train_dataset.loc[train_dataset['volunteer_rating'] < 4, features])
y_train_123 = np.array(train_dataset.loc[train_dataset['volunteer_rating'] < 4, 'volunteer_rating'],dtype=np.uint8)
print("Train Label distribution:",np.unique(y_train_123, return_counts=True))
X_test_123 = np.array(val_dataset.loc[val_dataset['volunteer_rating'] < 4, features])
y_test_123 = np.array(val_dataset.loc[val_dataset['volunteer_rating'] < 4, 'volunteer_rating'],dtype=np.uint8)
print("Val Label distribution:",np.unique(y_test_123, return_counts=True))

scaler = preprocessing.StandardScaler()
scaler.fit(X_train_123)
X_train_123 = scaler.fit_transform(X_train_123)
X_test_123 = scaler.transform(X_test_123)

In [ ]:
X_res_123, y_res_123 = ros.fit_resample(X_train_123, y_train_123)

clf = lgb.LGBMClassifier(random_state=42)
clf = clf.fit(X_res_123, y_res_123)
predict_labels = clf.predict(X_test_123)
predict_proba = clf.predict_proba(X_test_123)
acc = accuracy_score(y_test_123, predict_labels)
f1 = f1_score(y_test_123, predict_labels, average="macro")
roc = roc_auc_score(y_test_123, predict_proba, multi_class="ovr", average="macro")
print(f'acc: {acc:.3f}, f1: {f1:.3f}, roc: {roc:.3f}')

cm = confusion_matrix(y_test_123, predict_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=clf.classes_)
disp.plot()
plt.show()

importance_map = pd.DataFrame({'Feature' : features, 'Value' : clf.feature_importances_})

plt.figure(figsize=(20, 10))
sns.barplot(x="Value", y="Feature", data=importance_map.sort_values(by="Value", ascending=False)[:30])
plt.title('LightGBM Features')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()


Use dist/time as proxy, regression problem

In [ ]:
base['published_at'] = pd.to_datetime(base['published_at'])
base['completed_at'] = pd.to_datetime(base['completed_at'])
base['rescue_time_length'] = abs((base['completed_at'] - base['published_at'])).apply(lambda x: x.total_seconds())
base['distance_over_time'] = (base['user2donor'] + base['user2recipient']) / (base['rescue_time_length'])

In [ ]:
data = base[features + ['published_at','distance_over_time']]
data['distance_over_time'] = data['distance_over_time']*1e4
data = data.dropna(subset=['distance_over_time'])
data = data.fillna(data.median())

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# build model!
train_val_split = datetime(2020, 1, 1, 0, 0, 0)
data['published_at'] = pd.to_datetime(data['published_at'])
train_dataset = data[data['published_at'] < train_val_split]
val_dataset = data[data['published_at'] >= train_val_split]
train_dataset = train_dataset[features + ['published_at','distance_over_time']]
val_dataset = val_dataset[features + ['published_at','distance_over_time']]

In [ ]:
data = val_dataset['distance_over_time']
 
fig = plt.figure(figsize =(3, 5))
 
# Creating plot
plt.boxplot(data)
plt.xticks([1], ['test'])
plt.ylabel('Test distance over time')
 
# show plot
plt.show()

In [ ]:
# train, test set
X_train = np.array(train_dataset[features])
y_train = np.array(train_dataset['distance_over_time'],dtype=np.uint8)
print("Train Label distribution:")
#train_dataset['distance_over_seconds'].plot.hist()
X_test = np.array(val_dataset[features])
y_test = np.array(val_dataset['distance_over_time'],dtype=np.uint8)
print("Val Label distribution:")
# val_dataset['distance_over_seconds'].plot.hist()

In [ ]:
# 1) Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# 2) Scale
scaler = preprocessing.StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ------------------ FIX START ------------------
# Ensure the length of 'features' matches X_train's columns:
actual_num_columns = X_train.shape[1]
if len(features) != actual_num_columns:
    features = features[:actual_num_columns]
# ------------------ FIX END --------------------

# 3) LightGBM
models = {'lgb': lgb.LGBMRegressor(random_state=42, reg_alpha=1)}
for model_name, model in models.items():
    model.fit(X_train, y_train)
    predict_labels = model.predict(X_test)
    mse = mean_squared_error(y_test, predict_labels)
    r2 = r2_score(y_test, predict_labels)
    print(model_name, f'mse: {mse:.3f}, r2: {r2:.3f}')

# 4) Lasso
reduced_model = Lasso(alpha=0.1)
reduced_model.fit(X_train, y_train)
preserved_features = np.array(features)[(reduced_model.coef_ != 0)]
reduced_predict_labels = reduced_model.predict(X_test)
reduced_mse = mean_squared_error(y_test, reduced_predict_labels)
reduced_r2 = r2_score(y_test, reduced_predict_labels)

print('reduced model features', preserved_features)
print(f'mse: {reduced_mse:.3f}, r2: {reduced_r2:.3f}')


In [ ]:
sns.regplot(x="user2recipient", y="distance_over_time", data=train_dataset)

In [ ]:
importance_map = pd.DataFrame({'Feature' : features, 'Value' : model.feature_importances_})

plt.figure(figsize=(20, 10))
sns.barplot(x="Value", y="Feature", data=importance_map.sort_values(by="Value", ascending=False)[:30])
plt.title('LightGBM Features')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
data.shape